In [1]:
# Annuity Calculator
# Computes the accumulated value of:
#   - Simple or general annuities (entered via GUI).
#       * Simple  annuity: payment frequency == compounding frequency
#       * General annuity: payment frequency != compounding frequency
#         (see Section 4.7 of the Financial Math text). The per-period
#         payment rate is converted from the compounding rate via
#         j = (1 + i_c)^(c/p) - 1,
#         where c = compounding freq/year and p = payment freq/year.
#   - Block annuities (multiple interest rate blocks, loaded from CSV).
#
# CSV format for block annuities -- one row per block:
#   interest_rate_per_period, payment_per_period, number_of_periods
#
# A block with payment = 0 is valid and models a period where no new
# payments are made but the existing balance continues to earn interest.

import csv
import tkinter as tk
from tkinter import ttk, filedialog, messagebox


# -- Core calculation classes and functions ----------------------------------

class Block:
    """Represents one block of a savings annuity with a fixed interest rate per period.

    Attributes:
        i  -- interest rate per period
        r  -- payment made at the end of each period
        n  -- number of periods in this block
        v  -- accumulated value (computed by acc_value / fin_value)
    """

    def __init__(self, i, r, n):
        self.i = i
        self.r = r
        self.n = n
        self.v = 0

    def acc_value(self):
        """Compute the accumulated value of this block at the end of its own term.

        Uses the ordinary annuity formula: v = R * [(1+i)^n - 1] / i
        Special case: when i = 0, the accumulated value is simply R * n.
        """
        if self.i == 0:
            self.v = self.r * self.n
        else:
            g = (1 + self.i) ** self.n
            self.v = self.r * (g - 1) / self.i
        return self.v

    def fin_value(self, i_new, n_new):
        """Compound this block's value forward through a subsequent block.

        After a block ends, its accumulated value continues to earn interest
        at the rates of all subsequent blocks. This method must be called once
        for each subsequent block.

        Args:
            i_new  -- interest rate per period of the subsequent block
            n_new  -- number of periods in the subsequent block
        """
        self.v = self.v * (1 + i_new) ** n_new
        return self.v


def compute_blocks(blocks):
    """Compute accumulated and final values for a list of Block objects.

    Returns:
        acc_values   -- list of each block's value at the end of its own term
        final_values -- list of each block's value at the end of the full term
        total        -- sum of all final values
    """
    n = len(blocks)
    V = [b.acc_value() for b in blocks]
    acc_values = list(V)

    # Compound each block forward through all subsequent blocks
    for k in range(n - 1):
        for j in range(k + 1, n):
            V[k] = blocks[k].fin_value(blocks[j].i, blocks[j].n)

    return acc_values, V, sum(V)


def payment_period_rate(nominal_annual_rate, compounding_freq, payment_freq):
    """Return the effective interest rate per payment period.

    If compounding_freq == payment_freq (simple annuity) this is just
    nominal_annual_rate / payment_freq. Otherwise it is the equivalent
    payment-period rate derived from the per-compounding-period rate:

        j = (1 + i_c)^(c / p) - 1

    where i_c = nominal_annual_rate / c, c = compounding_freq,
    p = payment_freq.
    """
    if compounding_freq == payment_freq:
        return nominal_annual_rate / payment_freq
    i_c = nominal_annual_rate / compounding_freq
    return (1 + i_c) ** (compounding_freq / payment_freq) - 1


def solve_simple_annuity(j, S=0.0, R=0.0, n=0.0):
    """Solve for S, R, or n in an ordinary, simple/general annuity.

    Exactly one of S, R, n must be zero; that variable is computed.

    Formulas (per-payment-period rate j):
        S = R * [(1 + j)^n - 1] / j
        R = S * j / [(1 + j)^n - 1]
        n = ln(1 + S*j/R) / ln(1 + j)

    Special case j == 0:  S = R*n, R = S/n, n = S/R.

    Returns (kind, value, text) where kind is one of 'S', 'R', 'n', 'error'.
    """
    import math

    if j < 0:
        return ("error", None, "Rate j must be >= 0.")
    missing = sum(1 for v in (S, R, n) if v == 0)
    if missing != 1:
        return ("error", None,
                "Leave exactly one of S, R, or n at 0 "
                f"(currently {missing} are blank).")
    if S < 0 or R < 0 or n < 0:
        return ("error", None, "S, R, n must be non-negative.")

    # Zero-rate branch
    if j == 0:
        if S == 0:
            return ("S", R * n, f"${R * n:,.2f}")
        if R == 0:
            if n == 0:
                return ("error", None, "Cannot solve: n = 0.")
            return ("R", S / n, f"${S / n:,.2f}")
        # n == 0
        if R == 0:
            return ("error", None, "Cannot solve: R = 0.")
        return ("n", S / R, f"{S / R:,.4f}")

    g = (1 + j) ** n if n > 0 else None

    if S == 0:
        S_val = R * (g - 1) / j
        return ("S", S_val, f"${S_val:,.2f}")

    if R == 0:
        if g == 1:
            return ("error", None, "Cannot solve for R: (1+j)^n = 1.")
        R_val = S * j / (g - 1)
        return ("R", R_val, f"${R_val:,.2f}")

    # n == 0  -> solve for n
    if R <= 0:
        return ("error", None, "R must be > 0 to solve for n.")
    ratio = 1 + S * j / R
    if ratio <= 0:
        return ("error", None, "Infeasible inputs for solving n.")
    n_val = math.log(ratio) / math.log(1 + j)
    return ("n", n_val, f"{n_val:,.4f} periods")


# -- GUI application ----------------------------------------------------------

class AnnuityCalculatorApp:

    FREQ_OPTIONS = [
        ("Annual (1)",       1),
        ("Semi-annual (2)",  2),
        ("Quarterly (4)",    4),
        ("Monthly (12)",    12),
        ("Weekly (52)",     52),
        ("Daily (365)",    365),
    ]

    # Default label/value for both frequency dropdowns
    DEFAULT_FREQ_LABEL = "Monthly (12)"
    DEFAULT_FREQ_VAL   = 12

    def __init__(self, root):
        self.root = root
        self.root.title("Annuity Calculator")
        self.root.resizable(False, False)
        root.lift()
        root.attributes('-topmost', True)

        notebook = ttk.Notebook(root)
        notebook.pack(fill="both", expand=True, padx=12, pady=12)

        simple_frame = ttk.Frame(notebook, padding=12)
        notebook.add(simple_frame, text="  Simple Annuity  ")
        self._build_simple_tab(simple_frame)

        block_frame = ttk.Frame(notebook, padding=12)
        notebook.add(block_frame, text="  Block Annuity  ")
        self._build_block_tab(block_frame)

    # -- Simple Annuity tab ---------------------------------------------------

    def _build_simple_tab(self, frame):
        # -- Instruction --
        ttk.Label(frame,
                  text="Leave exactly one of S, R, or n at 0 to have it computed.",
                  foreground="grey").grid(row=0, column=0, sticky="w", pady=(0, 6))

        # -- Inputs --
        inp = ttk.LabelFrame(frame, text="Inputs", padding=12)
        inp.grid(row=1, column=0, sticky="ew", pady=(0, 10))

        ttk.Label(inp, text="Nominal Annual Rate (%):").grid(
            row=0, column=0, sticky="w", pady=5)
        self.s_rate = tk.StringVar()
        ttk.Entry(inp, textvariable=self.s_rate, width=20).grid(
            row=0, column=1, padx=12, pady=5)

        freq_labels = [label for label, _ in self.FREQ_OPTIONS]

        # Payment frequency
        ttk.Label(inp, text="Payment Frequency:").grid(
            row=1, column=0, sticky="w", pady=5)
        self.s_pay_freq_val = tk.IntVar(value=self.DEFAULT_FREQ_VAL)
        self.s_pay_freq_label = tk.StringVar(value=self.DEFAULT_FREQ_LABEL)
        pay_cb = ttk.Combobox(inp, textvariable=self.s_pay_freq_label,
                              values=freq_labels, state="readonly", width=18)
        pay_cb.grid(row=1, column=1, padx=12, pady=5)
        pay_cb.bind("<<ComboboxSelected>>",
                    lambda e: self._sync_freq(self.s_pay_freq_label,
                                              self.s_pay_freq_val))

        # Compounding frequency
        ttk.Label(inp, text="Compounding Frequency:").grid(
            row=2, column=0, sticky="w", pady=5)
        self.s_cmp_freq_val = tk.IntVar(value=self.DEFAULT_FREQ_VAL)
        self.s_cmp_freq_label = tk.StringVar(value=self.DEFAULT_FREQ_LABEL)
        cmp_cb = ttk.Combobox(inp, textvariable=self.s_cmp_freq_label,
                              values=freq_labels, state="readonly", width=18)
        cmp_cb.grid(row=2, column=1, padx=12, pady=5)
        cmp_cb.bind("<<ComboboxSelected>>",
                    lambda e: self._sync_freq(self.s_cmp_freq_label,
                                              self.s_cmp_freq_val))

        ttk.Label(inp, text="Accumulated Value S ($):").grid(
            row=3, column=0, sticky="w", pady=5)
        self.s_fv = tk.StringVar(value="0")
        ttk.Entry(inp, textvariable=self.s_fv, width=20).grid(
            row=3, column=1, padx=12, pady=5)

        ttk.Label(inp, text="Payment per Period R ($):").grid(
            row=4, column=0, sticky="w", pady=5)
        self.s_pmt = tk.StringVar(value="0")
        ttk.Entry(inp, textvariable=self.s_pmt, width=20).grid(
            row=4, column=1, padx=12, pady=5)

        ttk.Label(inp, text="Number of Periods n:").grid(
            row=5, column=0, sticky="w", pady=5)
        self.s_nper = tk.StringVar(value="0")
        ttk.Entry(inp, textvariable=self.s_nper, width=20).grid(
            row=5, column=1, padx=12, pady=5)

        # -- Buttons --
        btn_row = ttk.Frame(frame)
        btn_row.grid(row=2, column=0, pady=8)
        ttk.Button(btn_row, text="Calculate", command=self._calc_simple).grid(
            row=0, column=0, padx=6)
        ttk.Button(btn_row, text="Clear", command=self._clear_simple).grid(
            row=0, column=1, padx=6)

        # -- Result --
        res = ttk.LabelFrame(frame, text="Result", padding=12)
        res.grid(row=3, column=0, sticky="ew")

        self.s_result = tk.StringVar(value="-")
        ttk.Label(res, textvariable=self.s_result,
                  font=("Arial", 13, "bold")).grid(row=0, column=0, pady=4)

        self.s_detail = tk.StringVar(value="")
        ttk.Label(res, textvariable=self.s_detail,
                  foreground="grey", font=("Arial", 9)).grid(row=1, column=0)

    def _sync_freq(self, label_var, int_var):
        """Sync an IntVar to match the current frequency label."""
        label = label_var.get()
        for lbl, val in self.FREQ_OPTIONS:
            if lbl == label:
                int_var.set(val)
                return

    def _calc_simple(self):
        try:
            annual_rate = float(self.s_rate.get()) / 100
            pay_freq    = self.s_pay_freq_val.get()
            cmp_freq    = self.s_cmp_freq_val.get()
            S           = float(self.s_fv.get())
            R           = float(self.s_pmt.get())
            n           = float(self.s_nper.get())
        except ValueError:
            messagebox.showerror("Input Error", "Please enter valid numeric values.")
            return

        j = payment_period_rate(annual_rate, cmp_freq, pay_freq)
        kind, value, text = solve_simple_annuity(j, S=S, R=R, n=n)

        if kind == "error":
            messagebox.showerror("Input Error", text)
            return

        # Echo the computed value back into its field
        if kind == "S":
            self.s_fv.set(f"{value:.2f}")
        elif kind == "R":
            self.s_pmt.set(f"{value:.2f}")
        elif kind == "n":
            self.s_nper.set(f"{value:.4f}")

        annuity_kind = "simple" if pay_freq == cmp_freq else "general"
        label_map = {
            "S": "Accumulated Value S:",
            "R": "Payment per Period R:",
            "n": "Number of Periods n:",
        }
        self.s_result.set(f"{label_map[kind]}   {text}")

        # Figures used/computed for the detail line
        S_used = value if kind == "S" else S
        R_used = value if kind == "R" else R
        n_used = value if kind == "n" else n
        detail = (
            f"{annuity_kind} annuity   |   "
            f"j = {j * 100:.6f}% per payment period   |   "
            f"n = {n_used:.4f}   |   R = ${R_used:,.2f}   |   S = ${S_used:,.2f}"
        )
        if annuity_kind == "general":
            i_c = annual_rate / cmp_freq
            detail += f"\n(derived from per-compounding-period rate i = {i_c*100:.6f}%)"
        self.s_detail.set(detail)

    def _clear_simple(self):
        """Reset all Simple Annuity inputs and results to their defaults."""
        self.s_rate.set("")
        self.s_fv.set("0")
        self.s_pmt.set("0")
        self.s_nper.set("0")
        self.s_pay_freq_label.set(self.DEFAULT_FREQ_LABEL)
        self.s_pay_freq_val.set(self.DEFAULT_FREQ_VAL)
        self.s_cmp_freq_label.set(self.DEFAULT_FREQ_LABEL)
        self.s_cmp_freq_val.set(self.DEFAULT_FREQ_VAL)
        self.s_result.set("-")
        self.s_detail.set("")

    # -- Block Annuity tab ---------------------------------------------------

    def _build_block_tab(self, frame):
        # -- File selection --
        fsel = ttk.LabelFrame(frame, text="CSV Input File", padding=12)
        fsel.grid(row=0, column=0, sticky="ew", pady=(0, 10))

        self.b_filepath = tk.StringVar()
        ttk.Entry(fsel, textvariable=self.b_filepath, width=44).grid(
            row=0, column=0, padx=(0, 8))
        ttk.Button(fsel, text="Browse...", command=self._browse_csv).grid(
            row=0, column=1)

        ttk.Label(fsel,
                  text="CSV format per row:  "
                       "interest_rate_per_period,  payment_per_period,  num_periods",
                  foreground="grey", font=("Arial", 8)).grid(
            row=1, column=0, columnspan=2, sticky="w", pady=(8, 0))

        # -- Buttons --
        btn_row = ttk.Frame(frame)
        btn_row.grid(row=1, column=0, pady=8)
        ttk.Button(btn_row, text="Calculate", command=self._calc_block).grid(
            row=0, column=0, padx=6)
        ttk.Button(btn_row, text="Clear", command=self._clear_block).grid(
            row=0, column=1, padx=6)

        # -- Results --
        res = ttk.LabelFrame(frame, text="Results", padding=12)
        res.grid(row=2, column=0, sticky="nsew")

        self.b_results = tk.Text(
            res, height=16, width=58,
            font=("Courier New", 10), state="disabled",
            bg="#f8f8f8", relief="flat")
        self.b_results.grid(row=0, column=0)

        sb = ttk.Scrollbar(res, command=self.b_results.yview)
        sb.grid(row=0, column=1, sticky="ns")
        self.b_results.config(yscrollcommand=sb.set)

    def _browse_csv(self):
        path = filedialog.askopenfilename(
            title="Select block annuity CSV file",
            filetypes=[("CSV files", "*.csv"), ("All files", "*.*")])
        if path:
            self.b_filepath.set(path)

    def _calc_block(self):
        path = self.b_filepath.get().strip()
        if not path:
            messagebox.showerror("Input Error", "Please select a CSV file.")
            return

        try:
            B = []
            with open(path) as f:
                for row in csv.reader(f):
                    if row:
                        B.append(Block(float(row[0]), float(row[1]), float(row[2])))
        except Exception as e:
            messagebox.showerror("File Error", f"Could not read CSV file:\n{e}")
            return

        if not B:
            messagebox.showerror("File Error", "CSV file is empty.")
            return

        acc_vals, final_vals, total = compute_blocks(B)

        lines = []
        lines.append(f"{'Block':<7} {'Rate/Period':>13} {'Payment':>12} {'Periods':>8}")
        lines.append("-" * 44)
        for k, b in enumerate(B):
            pmt_str = f"${b.r:>10,.2f}" if b.r > 0 else f"{'-':>11}"
            lines.append(
                f"{k:<7} {b.i:>13.6f}  {pmt_str}  {int(b.n):>6}")

        lines.append("")
        lines.append("Accumulated value at end of each block:")
        for k, v in enumerate(acc_vals):
            lines.append(f"  Block {k}:  ${v:>14,.2f}")

        if len(B) > 1:
            lines.append("")
            lines.append("Value of each block at end of full term:")
            for k, v in enumerate(final_vals):
                lines.append(f"  Block {k}:  ${v:>14,.2f}")

        lines.append("")
        lines.append("-" * 44)
        lines.append(f"  Total:    ${total:>14,.2f}")

        self._set_block_results("\n".join(lines))

    def _set_block_results(self, text):
        self.b_results.config(state="normal")
        self.b_results.delete("1.0", tk.END)
        self.b_results.insert(tk.END, text)
        self.b_results.config(state="disabled")

    def _clear_block(self):
        """Reset the Block Annuity tab: clear the CSV path and results."""
        self.b_filepath.set("")
        self._set_block_results("")


# -- Entry point --------------------------------------------------------------

if __name__ == "__main__":
    root = tk.Tk()
    AnnuityCalculatorApp(root)
    root.mainloop()
